# Credit Risk Decision Engine — Deployable Model Comparison

Logistic Regression, XGBoost, and LightGBM use identical deployable features, stratified folds, and fold-fitted preprocessing. Model selection never uses the policy holdout or final test.


In [1]:
from pathlib import Path
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
from sklearn.model_selection import train_test_split

from src.components.data_loader import load_training_data
from src.components.data_preprocessor import select_production_raw_features
from src.config import POLICY_SIZE, RANDOM_STATE, TARGET_COLUMN, TEST_SIZE
from src.pipeline.training_pipeline import compare_candidate_models


## Create model, policy, and untouched final-test partitions


In [2]:
df = load_training_data()
raw = select_production_raw_features(df)
development_x, reserved_test_x, development_y, reserved_test_y = train_test_split(
    raw, df[TARGET_COLUMN], test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df[TARGET_COLUMN]
)
model_x, policy_x, model_y, policy_y = train_test_split(
    development_x, development_y, test_size=POLICY_SIZE, random_state=RANDOM_STATE, stratify=development_y
)
selected_name, selected_parameters, comparison = compare_candidate_models(model_x, model_y)
comparison_table = pd.DataFrame(comparison).T
display(comparison_table[["available", "roc_auc", "pr_auc", "brier_score", "training_seconds"]].sort_values("roc_auc", ascending=False))
print("Selected production candidate:", selected_name)
print("Policy and final-test rows not used for model selection:", len(policy_x), len(reserved_test_x))


,available,roc_auc,pr_auc,brier_score,training_seconds
XGBoost,True,0.684041,0.163565,0.071458,18.334335
LightGBM,True,0.680292,0.160244,0.071649,14.475063
Logistic Regression,True,0.640067,0.132286,0.072782,4.925131


Selected production candidate: xgboost
Policy and final-test rows not used for model selection: 49202 61503


The winner is selected dynamically by development out-of-fold ROC-AUC with PR-AUC as tie-breaker. A previous full-feature XGBoost result is retained only as benchmark context; no candidate is assumed to remain best after removing unavailable features.
